# Train Grasp-Anything++ trên Google Colab

Notebook này luôn đồng bộ code mới nhất từ branch `text-image-aware-v2`. Dataset, split, Hugging Face cache, log và checkpoint được lưu trên Google Drive. Sau khi dataset đã dựng xong, chạy lại notebook sẽ tái sử dụng cache và không download data lần nữa. Mặc định notebook chạy tuần tự ba arm `notext / noalign / full`, giữ riêng validation để chọn checkpoint và test độc lập để báo cáo Seen / Unseen / harmonic mean H.

In [6]:
#@title 1. Cấu hình — chỉnh mọi tham số chính tại đây
REPO_URL = "https://github.com/duncan-nguyen/QACI-HW.git"
BRANCH = "text-image-aware-v2"
DRIVE_ROOT = "/content/drive/MyDrive/[Research Space]/[QACI] VLA HW"  #@param {type:"string"}
LOCAL_REPO = "/content/QACI-HW"
LOCAL_DATA_ROOT = "/content/qaci-data"

# Data. 2.000 scene phù hợp để smoke train; tăng dần khi đã kiểm tra pipeline.
N_SCENES = 100_000  #@param {type:"integer"}
PACK_MASKS = True  #@param {type:"boolean"}
IMAGES_FROM_ZIP = True  #@param {type:"boolean"}
DOWNLOAD_WORKERS = 16  #@param {type:"integer"}

# Train.
EPOCHS = 20  #@param {type:"integer"}
BATCHES_PER_EPOCH = 250  #@param {type:"integer"}
# VRAM đủ cho 128; RAM hệ thống mới là giới hạn của DataLoader.
BATCH_SIZE = 128  #@param {type:"integer"}
VAL_BATCH_SIZE = 64  #@param {type:"integer"}
EVAL_BATCH_SIZE = 64  #@param {type:"integer"}
N_QUALITATIVE_CANDIDATES = 4  #@param {type:"integer"}
NUM_WORKERS = 4  #@param {type:"integer"}
PREFETCH_FACTOR = 2  #@param {type:"integer"}
INPUT_SIZE = 224  #@param {type:"integer"}
# TRAIN_FRACTION áp lên phần dev còn lại sau khi đã giữ TEST_SPLIT.
# Mặc định: khoảng 99.0% train, 0.5% validation và 0.5% test.
TRAIN_FRACTION = 0.995  #@param {type:"number"}
TEST_SPLIT = 0.005  #@param {type:"number"}
RANDOM_SEED = 123  #@param {type:"integer"}
LEARNING_RATE = 0.002  #@param {type:"number"}
DESCRIPTION = "ga-pp-colab"  #@param {type:"string"}
ARMS = "notext,noalign,full"  #@param {type:"string"}
DIAG_INTERVAL = 100  #@param {type:"integer"}

# Model text-image-aware-v2.
W_ALIGN = 0.3  #@param {type:"number"}
WARMUP_EPOCHS = 3  #@param {type:"integer"}
ALIGN_MODE = "soft"  #@param ["soft", "hard"]
REGION_TEXT = 1
FUSION = "residual"
ALIGN_STAGE = "bottleneck"
AMP = "auto"  #@param ["auto", "off", "bf16", "fp16"]

ARM_CONFIGS = {
    "notext":  dict(label="No text", use_text=0, w_align=0.0,
                    align_mode="soft", region_text=0, fusion="residual"),
    "noalign": dict(label="Text, no alignment loss", use_text=1, w_align=0.0,
                    align_mode="soft", region_text=1, fusion="residual"),
    "full":    dict(label="V2 full", use_text=1, w_align=W_ALIGN,
                    align_mode=ALIGN_MODE, region_text=REGION_TEXT, fusion=FUSION),
}
SELECTED_ARMS = [x.strip() for x in ARMS.split(",") if x.strip()]
unknown_arms = sorted(set(SELECTED_ARMS) - set(ARM_CONFIGS))
if unknown_arms:
    raise ValueError(f"Arm không hợp lệ: {unknown_arms}; chọn trong {list(ARM_CONFIGS)}")
if len(SELECTED_ARMS) != len(set(SELECTED_ARMS)):
    raise ValueError(f"ARMS chứa arm lặp: {SELECTED_ARMS}")

assert (N_SCENES > 0 and EPOCHS > 0 and BATCH_SIZE > 0
        and VAL_BATCH_SIZE > 0 and EVAL_BATCH_SIZE > 0
        and N_QUALITATIVE_CANDIDATES > 0 and PREFETCH_FACTOR > 0)
assert 0 < TRAIN_FRACTION < 1
assert 0 < TEST_SPLIT < 1
assert DIAG_INTERVAL <= BATCHES_PER_EPOCH

In [7]:
# 2. Mount Drive, clone/pull đúng branch và cài dependencies
import os
import signal
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

def run(cmd, *, cwd=None, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd), flush=True)
    try:
        subprocess.run(cmd, cwd=cwd, env=env, check=True)
    except subprocess.CalledProcessError as exc:
        if exc.returncode == -signal.SIGKILL:
            raise RuntimeError(
                "Tiến trình bị SIGKILL, thường do hết RAM hệ thống. "
                "Giảm NUM_WORKERS/PREFETCH_FACTOR trước, rồi giảm batch nếu cần.") from exc
        raise

repo = Path(LOCAL_REPO)
if repo.exists() and not (repo / ".git").is_dir():
    raise RuntimeError(f"{repo} đã tồn tại nhưng không phải Git repo")
if not repo.exists():
    run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL, repo])
else:
    # Mỗi lần chạy cell này đều lấy code mới nhất; local edits sẽ làm pull dừng an toàn.
    run(["git", "fetch", "origin", BRANCH], cwd=repo)
    run(["git", "checkout", BRANCH], cwd=repo)
    run(["git", "pull", "--ff-only", "origin", BRANCH], cwd=repo)

os.chdir(repo)
Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(Path(DRIVE_ROOT) / "huggingface")
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# pyrealsense2 chỉ dành cho camera robot, không cần cho train và dễ thiếu wheel trên Colab.
requirements = [x.strip() for x in Path("requirements.txt").read_text().splitlines()
                if x.strip() and not x.strip().startswith("pyrealsense2")]
run([sys.executable, "-m", "pip", "install", "-q", *requirements])

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=False,
).stdout.strip()
print("branch :", subprocess.check_output(["git", "branch", "--show-current"], text=True).strip())
print("commit :", subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], text=True).strip())
print("GPU    :", gpu or "không tìm thấy — hãy chọn Runtime > Change runtime type > GPU")
if not gpu:
    raise RuntimeError("Notebook train yêu cầu GPU")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
$ git fetch origin text-image-aware-v2
$ git checkout text-image-aware-v2
$ git pull --ff-only origin text-image-aware-v2
$ /usr/bin/python3 -m pip install -q numpy opencv-python matplotlib scikit-image imageio torch torchvision torchsummary tensorboardX Pillow transformers
branch : text-image-aware-v2
commit : 51f1aa3
GPU    : NVIDIA RTX PRO 6000 Blackwell Server Edition, 97887 MiB


In [8]:
# 3. Dựng/train trên SSD local; Drive chỉ giữ một file tar để tái sử dụng
cache_key = f"ga-pp-{N_SCENES}-scenes-{'packed' if PACK_MASKS else 'raw'}"
persistent_root = Path(DRIVE_ROOT)
local_root = Path(LOCAL_DATA_ROOT)
DATA_DIR = local_root / cache_key
ARCHIVES_DIR = DATA_DIR / "_archives"
DATA_TAR = persistent_root / "dataset-cache" / f"{cache_key}.tar"
SPLIT_DIR = persistent_root / "splits" / cache_key
LOG_DIR = persistent_root / "logs"
BUILD_DONE = DATA_DIR / ".build_complete"
LOCAL_READY = DATA_DIR / ".local_ready"
required_dirs = ["image", "grasp_instructions", "grasp_label_positive",
                 "part_mask", "scene_description"]

local_root.mkdir(parents=True, exist_ok=True)
DATA_TAR.parent.mkdir(parents=True, exist_ok=True)
local_free_gb = shutil.disk_usage(local_root).free / 1024**3
output_gb = 20 * N_SCENES / 100_000 if PACK_MASKS else 84 * N_SCENES / 100_000
need_local_gb = output_gb + (76 if IMAGES_FROM_ZIP else 11) + 5
print(f"SSD local: còn {local_free_gb:.1f} GiB; lần build đầu cần khoảng {need_local_gb:.0f} GiB")
if not DATA_TAR.is_file() and local_free_gb < need_local_gb:
    raise RuntimeError(
        f"Không đủ SSD local: cần ~{need_local_gb:.0f} GiB. "
        "Giảm N_SCENES hoặc dùng runtime có ổ local lớn hơn.")

cache_ready = LOCAL_READY.is_file() and all((DATA_DIR / x).is_dir() for x in required_dirs)
if not cache_ready and DATA_TAR.is_file():
    print(f"Khôi phục dataset từ một file tuần tự trên Drive: {DATA_TAR}")
    run(["tar", "-xf", DATA_TAR, "-C", local_root])
    LOCAL_READY.write_text("extracted\n")
    cache_ready = all((DATA_DIR / x).is_dir() for x in required_dirs)

# Tận dụng phần download dở của phiên bản notebook cũ thay vì tải lại từ đầu.
legacy_dir = persistent_root / "data" / cache_key
if not cache_ready and legacy_dir.is_dir() and not DATA_DIR.exists():
    print(f"Chuyển cache cũ/dở từ Drive sang SSD local: {legacy_dir}")
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    run(["rsync", "-a", f"{legacy_dir}/", f"{DATA_DIR}/"])
    cache_ready = BUILD_DONE.is_file() and all((DATA_DIR / x).is_dir() for x in required_dirs)

if not cache_ready:
    DATA_DIR.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "script/build_ga_pp_subset.py",
           "--out", DATA_DIR, "--zips-dir", ARCHIVES_DIR,
           "--scenes", N_SCENES, "--workers", DOWNLOAD_WORKERS]
    if PACK_MASKS:
        cmd.append("--pack-masks")
    if IMAGES_FROM_ZIP:
        cmd.append("--images-from-zip")
    run(cmd)
    BUILD_DONE.write_text(f"branch={BRANCH}\nscenes={N_SCENES}\n")
    LOCAL_READY.write_text("built\n")
elif not LOCAL_READY.is_file():
    LOCAL_READY.write_text("migrated\n")

if not DATA_TAR.is_file():
    # Một file lớn trên Drive nhanh hơn hàng trăm nghìn file nhỏ và ghi atomic.
    tmp_tar = DATA_TAR.with_suffix(".tar.part")
    print(f"Đóng gói cache để các runtime sau không download lại: {DATA_TAR}")
    run(["tar", "-cf", tmp_tar, "-C", local_root, cache_key])
    os.replace(tmp_tar, DATA_TAR)

seen_file, unseen_file = SPLIT_DIR / "seen.obj", SPLIT_DIR / "unseen.obj"
if seen_file.is_file() and unseen_file.is_file():
    print(f"Dùng lại split đã có: {SPLIT_DIR}")
else:
    SPLIT_DIR.mkdir(parents=True, exist_ok=True)
    run([sys.executable, "split/build_grasp_anything_pp.py",
         "--data-dir", DATA_DIR, "--out-dir", SPLIT_DIR])

n_samples = len(list((DATA_DIR / "grasp_label_positive").glob("*.pt")))
drive_free_gb = shutil.disk_usage(persistent_root).free / 1024**3
print(f"Dataset local: {DATA_DIR}")
print(f"Dataset: {n_samples:,} sample | Drive còn trống: {drive_free_gb:.1f} GiB")


SSD local: còn 149.0 GiB; lần build đầu cần khoảng 101 GiB
Dùng lại split đã có: /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/splits/ga-pp-100000-scenes-packed
Dataset local: /content/qaci-data/ga-pp-100000-scenes-packed
Dataset: 441,298 sample | Drive còn trống: 141.6 GiB


In [9]:
# 4. Train tuần tự các arm; cùng data, split, seed và sample budget
LOG_DIR.mkdir(parents=True, exist_ok=True)
common_train_cmd = [
    sys.executable, "train_network.py",
    "--dataset", "grasp-anything-pp",
    "--dataset-path", DATA_DIR,
    "--split-path", SPLIT_DIR,
    "--network", "grconvnet3_align",
    "--use-depth", 0, "--use-rgb", 1, "--seen", 1,
    "--input-size", INPUT_SIZE,
    "--split", TRAIN_FRACTION, "--test-split", TEST_SPLIT,
    "--random-seed", RANDOM_SEED,
    "--epochs", EPOCHS, "--batches-per-epoch", BATCHES_PER_EPOCH,
    "--batch-size", BATCH_SIZE, "--val-batch-size", VAL_BATCH_SIZE,
    "--num-workers", NUM_WORKERS, "--prefetch-factor", PREFETCH_FACTOR,
    "--lr", LEARNING_RATE, "--lr-schedule", "cosine",
    "--amp", AMP, "--channels-last", "auto",
    "--align-stage", ALIGN_STAGE,
    "--warmup-epochs", WARMUP_EPOCHS,
    "--diag-interval", DIAG_INTERVAL, "--probe-samples", 8,
    "--logdir", LOG_DIR,
]

budget = EPOCHS * BATCHES_PER_EPOCH * BATCH_SIZE
print(f"Arms: {', '.join(SELECTED_ARMS)}")
print(f"Budget mỗi arm: {EPOCHS} x {BATCHES_PER_EPOCH} x {BATCH_SIZE} = {budget:,} sample")
for arm in SELECTED_ARMS:
    cfg = ARM_CONFIGS[arm]
    arm_description = f"{DESCRIPTION}-{arm}"
    counterfactual_every = 0 if not cfg["use_text"] else 5
    train_cmd = common_train_cmd + [
        "--use-text", cfg["use_text"],
        "--w-align", cfg["w_align"],
        "--align-mode", cfg["align_mode"],
        "--region-text", cfg["region_text"],
        "--fusion", cfg["fusion"],
        "--counterfactual-every", counterfactual_every,
        "--description", arm_description,
    ]
    print(f"\n===== Train {arm}: {cfg['label']} =====")
    run(train_cmd)

Arms: notext, noalign, full
Budget mỗi arm: 20 x 250 x 128 = 640,000 sample

===== Train notext: No text =====
$ /usr/bin/python3 train_network.py --dataset grasp-anything-pp --dataset-path /content/qaci-data/ga-pp-100000-scenes-packed --split-path /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/splits/ga-pp-100000-scenes-packed --network grconvnet3_align --use-depth 0 --use-rgb 1 --seen 1 --input-size 224 --split 0.995 --test-split 0.005 --random-seed 123 --epochs 20 --batches-per-epoch 250 --batch-size 128 --val-batch-size 64 --num-workers 4 --prefetch-factor 2 --lr 0.002 --lr-schedule cosine --amp auto --channels-last auto --align-stage bottleneck --warmup-epochs 3 --diag-interval 100 --probe-samples 8 --logdir /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/logs --use-text 0 --w-align 0.0 --align-mode soft --region-text 0 --fusion residual --counterfactual-every 0 --description ga-pp-colab-notext

===== Train noalign: Text, no alignment loss =====
$ /usr/bin/python3 tra

In [10]:
# 5. Chọn checkpoint validation tốt nhất của từng arm
import re

def checkpoint_iou(path):
    match = re.search(r"iou_([0-9.]+)$", path.name)
    return float(match.group(1)) if match else -1.0

run_by_arm, best_by_arm = {}, {}
for arm in SELECTED_ARMS:
    suffix = f"_{DESCRIPTION}-{arm}"
    candidates = sorted(
        (p for p in LOG_DIR.iterdir() if p.is_dir() and p.name.endswith(suffix)),
        key=lambda p: p.stat().st_mtime,
    )
    if not candidates:
        raise FileNotFoundError(f"Không thấy run cho arm {arm!r} trong {LOG_DIR}")
    run_dir = candidates[-1]
    checkpoints = list(run_dir.glob("epoch_*"))
    if not checkpoints:
        raise FileNotFoundError(f"Không thấy checkpoint trong {run_dir}")
    run_by_arm[arm] = run_dir
    best_by_arm[arm] = max(checkpoints, key=checkpoint_iou)

print(f"{'arm':<10} {'val success':>12}  checkpoint")
for arm in SELECTED_ARMS:
    best = best_by_arm[arm]
    print(f"{arm:<10} {checkpoint_iou(best):12.4f}  {best}")


arm         val success  checkpoint
notext           0.2824  /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/logs/260901_1837_ga-pp-colab-notext/epoch_18_iou_0.2824
noalign          0.3447  /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/logs/260901_1847_ga-pp-colab-noalign/epoch_17_iou_0.3447
full             0.3661  /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/logs/260901_1902_ga-pp-colab-full/epoch_18_iou_0.3661


In [ ]:
# 6. Test độc lập từng arm trên Seen và Unseen, rồi xuất bảng ablation
import json
from datetime import datetime, timezone

def verify_checkpoint_args(arm):
    args_path = run_by_arm[arm] / "commandline_args.json"
    if not args_path.is_file():
        raise FileNotFoundError(f"Thiếu {args_path}; không xác minh được checkpoint")
    train_args = json.loads(args_path.read_text())
    expected = ARM_CONFIGS[arm]
    checks = {
        "split": (float(train_args.get("split", -1)), TRAIN_FRACTION),
        "test_split": (float(train_args.get("test_split", 0)), TEST_SPLIT),
        "random_seed": (int(train_args.get("random_seed", -1)), RANDOM_SEED),
        "epochs": (int(train_args.get("epochs", -1)), EPOCHS),
        "batches_per_epoch": (int(train_args.get("batches_per_epoch", -1)), BATCHES_PER_EPOCH),
        "batch_size": (int(train_args.get("batch_size", -1)), BATCH_SIZE),
        "input_size": (int(train_args.get("input_size", -1)), INPUT_SIZE),
        "lr": (float(train_args.get("lr", -1)), LEARNING_RATE),
        "use_text": (int(train_args.get("use_text", -1)), expected["use_text"]),
        "w_align": (float(train_args.get("w_align", -1)), expected["w_align"]),
        "align_mode": (train_args.get("align_mode"), expected["align_mode"]),
        "region_text": (int(train_args.get("region_text", -1)), expected["region_text"]),
        "fusion": (train_args.get("fusion"), expected["fusion"]),
        "align_stage": (train_args.get("align_stage"), ALIGN_STAGE),
    }
    mismatches = [f"{k}: checkpoint={got}, expected={want}"
                  for k, (got, want) in checks.items() if got != want]
    if mismatches:
        raise RuntimeError(
            f"Checkpoint arm {arm!r} không khớp notebook; chạy lại cell Train: "
            + "; ".join(mismatches))

def evaluate_test(arm, seen):
    label = "Seen" if seen else "Unseen"
    checkpoint = best_by_arm[arm]
    cmd = [
        sys.executable, "evaluate.py",
        "--dataset", "grasp-anything-pp",
        "--dataset-path", DATA_DIR,
        "--split-path", SPLIT_DIR,
        "--network", checkpoint,
        "--input-size", INPUT_SIZE,
        "--use-depth", 0, "--use-rgb", 1,
        "--seen", int(seen),
        "--split", TRAIN_FRACTION,
        "--test-split", TEST_SPLIT,
        "--subset", "test",
        "--random-seed", RANDOM_SEED,
        "--num-workers", NUM_WORKERS,
        "--batch-size", EVAL_BATCH_SIZE,
        "--iou-eval",
    ]
    cmd = [str(x) for x in cmd]
    print(f"\n== {arm} / {label} independent test ==")
    print("$", " ".join(cmd), flush=True)
    process = subprocess.Popen(
        cmd, cwd=repo, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    result = None
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end="")
        match = re.search(r"IOU Results:\s+(\d+)/(\d+)\s*=\s*([0-9.]+)", line)
        if match:
            result = {"correct": int(match.group(1)), "total": int(match.group(2)),
                      "success": float(match.group(3))}
    returncode = process.wait()
    if returncode != 0:
        raise subprocess.CalledProcessError(returncode, cmd)
    if result is None:
        raise RuntimeError(f"Không đọc được IOU Results cho {arm}/{label}")
    return result

ablation_results = {}
for arm in SELECTED_ARMS:
    verify_checkpoint_args(arm)
    seen = evaluate_test(arm, True)
    unseen = evaluate_test(arm, False)
    s, u = seen["success"], unseen["success"]
    h = 0.0 if s + u == 0 else 2 * s * u / (s + u)
    row = {"label": ARM_CONFIGS[arm]["label"],
           "checkpoint": str(best_by_arm[arm]), "seen": seen, "unseen": unseen,
           "harmonic_mean": h}
    ablation_results[arm] = row
    per_run = {
        "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
        "subset": "test", "train_fraction_of_dev": TRAIN_FRACTION,
        "test_fraction": TEST_SPLIT, "random_seed": RANDOM_SEED,
        "eval_batch_size": EVAL_BATCH_SIZE, **row,
    }
    (run_by_arm[arm] / "test_results.json").write_text(
        json.dumps(per_run, indent=2) + "\n")

aggregate = {
    "evaluated_at_utc": datetime.now(timezone.utc).isoformat(),
    "budget_samples_per_arm": budget, "test_fraction": TEST_SPLIT,
    "random_seed": RANDOM_SEED, "eval_batch_size": EVAL_BATCH_SIZE,
    "results": ablation_results,
}
aggregate_path = LOG_DIR / f"{DESCRIPTION}-ablation-test-results.json"
aggregate_path.write_text(json.dumps(aggregate, indent=2) + "\n")

print(f"\n{'arm':<10} {'Seen test':>12} {'Unseen test':>12} {'H':>12}")
for arm in SELECTED_ARMS:
    row = ablation_results[arm]
    print(f"{arm:<10} {row['seen']['success']:12.4f} "
          f"{row['unseen']['success']:12.4f} {row['harmonic_mean']:12.4f}")
print("Đã lưu:", aggregate_path)

print("\nPaste these lines into docs/latex/sec/results.tex:")
macro_prefix = {"notext": "NoText", "noalign": "NoAlign", "full": "Full"}
for arm in SELECTED_ARMS:
    row, prefix = ablation_results[arm], macro_prefix[arm]
    print(f"\\renewcommand{{\\{prefix}Seen}}{{{row['seen']['success']:.3f}}}")
    print(f"\\renewcommand{{\\{prefix}Unseen}}{{{row['unseen']['success']:.3f}}}")
    print(f"\\renewcommand{{\\{prefix}H}}{{{row['harmonic_mean']:.3f}}}")



== notext / Seen independent test ==
$ /usr/bin/python3 evaluate.py --dataset grasp-anything-pp --dataset-path /content/qaci-data/ga-pp-100000-scenes-packed --split-path /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/splits/ga-pp-100000-scenes-packed --network /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/logs/260901_1837_ga-pp-colab-notext/epoch_18_iou_0.2824 --input-size 224 --use-depth 0 --use-rgb 1 --seen 1 --split 0.995 --test-split 0.005 --subset test --random-seed 123 --num-workers 4 --batch-size 64 --iou-eval


INFO:root:CUDA detected. Running with GPU acceleration.
INFO:root:Loading Grasp-Anything-Pp Dataset...
INFO:root:Index splits: train 204,316 · val 1,027 · test 1,031
INFO:root:Evaluating subset 'test': 1031 samples
INFO:root:Evaluation batch size: 64
INFO:root:Done
INFO:root:
Evaluating model /content/drive/MyDrive/[Research Space]/[QACI] VLA HW/logs/260901_1837_ga-pp-colab-notext/epoch_18_iou_0.2824


In [ ]:
# 7. Figure định tính: cùng ảnh Unseen test, hai part thật, NoAlign vs Full
import json
from IPython.display import Image, display

missing = [arm for arm in ("noalign", "full") if arm not in best_by_arm]
if missing:
    raise RuntimeError(f"Cần checkpoint của {missing}; đặt ARMS chứa noalign,full")

QUALITATIVE_DIR = LOG_DIR / f"{DESCRIPTION}-qualitative-unseen-test"
qualitative_cmd = [
    sys.executable, "script/visualize_alignment_comparison.py",
    "--noalign-checkpoint", best_by_arm["noalign"],
    "--full-checkpoint", best_by_arm["full"],
    "--dataset-path", DATA_DIR,
    "--split-path", SPLIT_DIR,
    "--out-dir", QUALITATIVE_DIR,
    "--input-size", INPUT_SIZE,
    "--split", TRAIN_FRACTION,
    "--test-split", TEST_SPLIT,
    "--subset", "test",
    "--seen", 0,
    "--random-seed", RANDOM_SEED,
    "--n-candidates", N_QUALITATIVE_CANDIDATES,
]
run(qualitative_cmd, cwd=repo)

manifest = json.loads((QUALITATIVE_DIR / "manifest.json").read_text())
candidate_pngs = [Path(item["png"]) for item in manifest["figures"]]
if not candidate_pngs:
    raise RuntimeError(f"Không sinh được candidate figure trong {QUALITATIVE_DIR}")
for path in candidate_pngs:
    print(path.name)
    display(Image(filename=str(path), width=1100))
print("Chọn một candidate dễ đọc; bản PDF vector nằm cùng thư mục trên Drive.")


## Lưu ý

- Không xoá file `.build_complete` trong thư mục dataset trên Drive nếu muốn tái sử dụng cache. Nếu lần tải đầu bị ngắt, chỉ cần chạy lại cell data; downloader sẽ tiếp tục và bỏ qua các file đã hoàn tất.
- `IMAGES_FROM_ZIP=False` chỉ hợp lý cho subset nhỏ. Với 100.000 scene, giữ `True`; cần khoảng 100 GB SSD local trong lần build đầu nhưng tránh khoảng 100.000 HTTP request riêng lẻ.
- Đổi `N_SCENES` hoặc `PACK_MASKS` tự động tạo cache riêng, không trộn lẫn dataset cũ.
- Dataset được train từ SSD local. Drive chỉ giữ một file `dataset-cache/*.tar`, giúp các runtime sau khôi phục nhanh và không download lại từ Hugging Face.
- Cell train chạy tuần tự các arm trong `ARMS`; mặc định là `notext,noalign,full`, cùng data, seed, split và sample budget.
- Validation chỉ dùng để chọn checkpoint. Cell test dùng `--subset test` cho cả Seen và Unseen, lưu `test_results.json` cạnh từng checkpoint và một bảng tổng hợp trong `LOG_DIR`.
- Test mặc định dùng `EVAL_BATCH_SIZE=64`. Nếu CUDA OOM, giảm xuống 32 hoặc 16; metric không đổi vì hậu xử lý và IoU vẫn tính riêng từng ảnh.
- Cell qualitative không train lại. Nó dùng best NoAlign/Full checkpoint, đúng Unseen test subset, và chọn candidate chỉ theo khoảng cách giữa hai GT part masks—not theo model score.
- Đổi `ARMS`, `TRAIN_FRACTION`, `TEST_SPLIT`, `RANDOM_SEED` hoặc cấu hình arm bắt buộc phải train lại; cell test sẽ từ chối checkpoint không khớp.
- Cell test in sẵn ba dòng LaTeX để chép vào `docs/latex/sec/results.tex`.